# 🛒 Custo de Vida Brasil — Case de Margem no Varejo

## Contexto

A **Rede Sabor & Cia Supermercados** (empresa fictícia, estado de São Paulo) reportava faturamento estável, mas percebia uma queda preocupante na margem de lucro nos últimos meses. Este notebook documenta a investigação completa conduzida para entender essa percepção.

## Problema de negócio

O gestor da rede identificou uma possível queda de margem e precisava decidir se deveria agir (cortar custos, renegociar fornecedores, revisar preços) ou se a variação observada era normal. A decisão de investir recursos numa correção depende de confirmar se o problema realmente existe.

## Perguntas de negócio

- A margem da rede realmente caiu, ou é uma percepção distorcida?
- Custos de fornecedores subiram sem repasse ao preço final?
- Descontos, ruptura de estoque ou mix de produtos explicam a variação?
- A inflação (IPCA) pressionou descontos ou custos-base?
- Existe diferença relevante entre lojas, regiões ou canais de venda?

## Hipóteses testadas

| # | Hipótese |
|---|---|
| 1 | Desconto concentrado em produtos/períodos específicos |
| 2 | Marketing ineficiente (baixo investimento gerando queda) |
| 3 | Custo do fornecedor subindo sem repasse ao preço |
| 4 | Mudança de mix de produtos (alta → baixa margem) |
| 5 | Ruptura de estoque causando reposição emergencial mais cara |
| 6 | Logística/frete mais caro em lojas do interior |
| 7 | Repasse de inflação (IPCA) via aumento de desconto |
| 8 | Canal de venda (Delivery/App vs. Loja Física) impactando margem |
| 9 | Sazonalidade (datas comemorativas afetando margem) |

## Dados necessários

- **Dados internos (fictícios):** vendas, estoque, marketing, lojas, categorias e produtos — 12 meses de operação, 10 lojas, 32 produtos em 8 categorias
- **Dados públicos reais:** IPCA via API SIDRA/IBGE (indicador de inflação)

> ⚠️ Os dados operacionais são sintéticos, gerados para fins educacionais. O IPCA é real e público. Essa combinação híbrida está documentada de forma transparente — mais detalhes na seção de limitações do README do projeto.

---

## 1. Preparação dos dados (RAW → PROCESSED)

As células abaixo constroem as dimensões (lojas, produtos), os fatos (vendas, estoque, marketing), validam qualidade (nulos, duplicidades, tipos) e enriquecem a tabela de vendas via merges dimensionais.

In [ ]:
# ==== IMPORTS ====
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import os

# ==== SEED (reprodutibilidade) ====
np.random.seed(42)

# ==== ESTRUTURA DE PASTAS ====
BASE_DIR = "/content/custo-vida-brasil"
RAW_DIR = f"{BASE_DIR}/data/raw"
PROCESSED_DIR = f"{BASE_DIR}/data/processed"
ANALYTICS_DIR = f"{BASE_DIR}/data/analytics"

for d in [RAW_DIR, PROCESSED_DIR, ANALYTICS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Estrutura criada com sucesso.")

Estrutura criada com sucesso.


In [ ]:
# ==== DIMENSÃO: LOJAS ====
lojas = pd.DataFrame({
    "loja_id": [f"L{i:02d}" for i in range(1, 11)],
    "nome_loja": [f"Sabor & Cia - Unidade {i}" for i in range(1, 11)],
    "regiao": [
        "Capital", "Capital", "Capital",
        "Grande SP", "Grande SP",
        "Interior", "Interior", "Interior",
        "Interior", "Litoral"
    ],
    "cidade": [
        "São Paulo", "São Paulo", "São Paulo",
        "Guarulhos", "Osasco",
        "Campinas", "Sorocaba", "Ribeirão Preto",
        "São José do Rio Preto", "Santos"
    ]
})

# ==== DIMENSÃO: CATEGORIAS ====
categorias = pd.DataFrame({
    "categoria_id": [f"C{i:02d}" for i in range(1, 9)],
    "categoria": [
        "Mercearia", "Hortifruti", "Açougue", "Laticínios",
        "Limpeza", "Higiene", "Bebidas", "Padaria"
    ]
})

lojas.to_parquet(f"{RAW_DIR}/dim_lojas.parquet", index=False)
categorias.to_parquet(f"{RAW_DIR}/dim_categorias.parquet", index=False)

print(lojas)
print(categorias)

  loja_id                 nome_loja     regiao                 cidade
0     L01   Sabor & Cia - Unidade 1    Capital              São Paulo
1     L02   Sabor & Cia - Unidade 2    Capital              São Paulo
2     L03   Sabor & Cia - Unidade 3    Capital              São Paulo
3     L04   Sabor & Cia - Unidade 4  Grande SP              Guarulhos
4     L05   Sabor & Cia - Unidade 5  Grande SP                 Osasco
5     L06   Sabor & Cia - Unidade 6   Interior               Campinas
6     L07   Sabor & Cia - Unidade 7   Interior               Sorocaba
7     L08   Sabor & Cia - Unidade 8   Interior         Ribeirão Preto
8     L09   Sabor & Cia - Unidade 9   Interior  São José do Rio Preto
9     L10  Sabor & Cia - Unidade 10    Litoral                 Santos
  categoria_id   categoria
0          C01   Mercearia
1          C02  Hortifruti
2          C03     Açougue
3          C04  Laticínios
4          C05     Limpeza
5          C06     Higiene
6          C07     Bebidas
7          C08

In [ ]:
# ==== DIMENSÃO: PRODUTOS ====
# Cada categoria terá alguns produtos representativos, com preço e custo base

produtos_por_categoria = {
    "C01": [("Arroz 5kg", 24.90, 16.50), ("Feijão 1kg", 8.90, 5.80), ("Açúcar 1kg", 4.50, 2.90), ("Óleo de Soja 900ml", 7.90, 5.20)],
    "C02": [("Banana Prata kg", 5.90, 3.20), ("Tomate kg", 7.50, 4.10), ("Alface unid", 2.50, 1.20), ("Batata kg", 4.90, 2.80)],
    "C03": [("Picanha kg", 59.90, 42.00), ("Frango kg", 12.90, 8.50), ("Linguiça kg", 18.90, 12.30), ("Carne Moída kg", 28.90, 19.50)],
    "C04": [("Leite 1L", 5.20, 3.60), ("Queijo Mussarela kg", 32.90, 22.00), ("Iogurte 170g", 2.80, 1.70), ("Manteiga 200g", 9.90, 6.50)],
    "C05": [("Detergente 500ml", 2.90, 1.60), ("Sabão em Pó 1kg", 12.90, 8.20), ("Desinfetante 1L", 6.90, 4.10), ("Água Sanitária 1L", 4.50, 2.60)],
    "C06": [("Sabonete unid", 2.50, 1.30), ("Shampoo 350ml", 14.90, 9.20), ("Papel Higiênico 12un", 19.90, 13.50), ("Creme Dental 90g", 4.90, 2.90)],
    "C07": [("Refrigerante 2L", 8.90, 5.60), ("Cerveja Lata", 3.90, 2.40), ("Água Mineral 1,5L", 3.50, 1.90), ("Suco 1L", 6.90, 4.30)],
    "C08": [("Pão Francês kg", 12.90, 6.50), ("Bolo Fatia", 5.90, 2.80), ("Rosquinha 300g", 7.90, 4.20), ("Torta Salgada", 9.90, 5.50)],
}

produtos_rows = []
pid = 1
for cat_id, itens in produtos_por_categoria.items():
    for nome, preco, custo in itens:
        produtos_rows.append({
            "produto_id": f"P{pid:03d}",
            "categoria_id": cat_id,
            "nome_produto": nome,
            "preco_base": preco,
            "custo_base": custo
        })
        pid += 1

produtos = pd.DataFrame(produtos_rows)
produtos.to_parquet(f"{RAW_DIR}/dim_produtos.parquet", index=False)

print(produtos.shape)
produtos.head(10)

(32, 5)


,produto_id,categoria_id,nome_produto,preco_base,custo_base
0,P001,C01,Arroz 5kg,24.9,16.5
1,P002,C01,Feijão 1kg,8.9,5.8
2,P003,C01,Açúcar 1kg,4.5,2.9
3,P004,C01,Óleo de Soja 900ml,7.9,5.2
4,P005,C02,Banana Prata kg,5.9,3.2
5,P006,C02,Tomate kg,7.5,4.1
6,P007,C02,Alface unid,2.5,1.2
7,P008,C02,Batata kg,4.9,2.8
8,P009,C03,Picanha kg,59.9,42.0
9,P010,C03,Frango kg,12.9,8.5


In [ ]:
# ==== CONFIGURAÇÃO TEMPORAL ====
data_inicio = datetime(2024, 1, 1)
data_fim = datetime(2024, 12, 31)
datas = pd.date_range(data_inicio, data_fim, freq="D")

canais = ["Loja Física", "Delivery/App"]

vendas_rows = []

for _, loja in lojas.iterrows():
    # fator de sazonalidade/volume específico por loja
    fator_loja = np.random.uniform(0.7, 1.3)

    for _, prod in produtos.iterrows():
        # probabilidade base de venda do produto naquele loja, por dia
        prob_venda_dia = np.random.uniform(0.4, 0.9)

        for data in datas:
            if np.random.rand() > prob_venda_dia:
                continue  # não vendeu esse produto nessa loja nesse dia

            # sazonalidade mensal leve
            mes = data.month
            fator_sazonal = 1 + 0.15 * np.sin(2 * np.pi * mes / 12)

            qtd = max(1, int(np.random.poisson(6) * fator_loja * fator_sazonal))

            # preço praticado (com variação de precificação ao longo do ano)
            variacao_preco = np.random.uniform(0.95, 1.25)
            preco_praticado = round(prod["preco_base"] * variacao_preco, 2)

            # desconto aplicado (%)
            desconto_pct = round(np.random.choice(
                [0, 0, 0, 0.05, 0.10, 0.15, 0.20, 0.30],
                p=[0.5, 0.1, 0.1, 0.1, 0.08, 0.06, 0.04, 0.02]
            ), 2)

            preco_final = round(preco_praticado * (1 - desconto_pct), 2)
            canal = np.random.choice(canais, p=[0.8, 0.2])

            vendas_rows.append({
                "data": data,
                "loja_id": loja["loja_id"],
                "produto_id": prod["produto_id"],
                "quantidade": qtd,
                "preco_praticado": preco_praticado,
                "desconto_pct": desconto_pct,
                "preco_final": preco_final,
                "canal": canal
            })

fato_vendas = pd.DataFrame(vendas_rows)
fato_vendas.to_parquet(f"{RAW_DIR}/fato_vendas.parquet", index=False)

print(fato_vendas.shape)
fato_vendas.head(10)

(76688, 8)


,data,loja_id,produto_id,quantidade,preco_praticado,desconto_pct,preco_final,canal
0,2024-01-01,L01,P001,2,30.13,0.00,30.13,Loja Física
1,2024-01-02,L01,P001,4,27.57,0.00,27.57,Loja Física
2,2024-01-03,L01,P001,4,27.50,0.00,27.50,Loja Física
3,2024-01-04,L01,P001,5,28.77,0.00,28.77,Loja Física
4,2024-01-05,L01,P001,3,27.54,0.00,27.54,Loja Física
5,2024-01-07,L01,P001,5,26.09,0.00,26.09,Loja Física
6,2024-01-08,L01,P001,4,31.03,0.05,29.48,Loja Física
7,2024-01-09,L01,P001,5,30.10,0.00,30.10,Loja Física
8,2024-01-10,L01,P001,5,28.98,0.05,27.53,Loja Física
9,2024-01-11,L01,P001,3,23.89,0.00,23.89,Loja Física


In [ ]:
# ==== FATO: ESTOQUE MENSAL + RUPTURA ====
meses = pd.date_range(data_inicio, data_fim, freq="MS")  # início de cada mês

estoque_rows = []

for _, loja in lojas.iterrows():
    for _, prod in produtos.iterrows():
        # nível de estoque médio "alvo" pra esse produto nessa loja
        estoque_alvo = np.random.randint(30, 150)

        for mes in meses:
            estoque_inicial = max(0, int(np.random.normal(estoque_alvo, estoque_alvo * 0.25)))

            # dias em ruptura no mês (0 a X, com maior chance de ser baixo)
            dias_ruptura = int(np.random.choice(
                [0, 1, 2, 3, 4, 5, 8, 12],
                p=[0.45, 0.15, 0.12, 0.10, 0.08, 0.05, 0.03, 0.02]
            ))

            estoque_rows.append({
                "mes_referencia": mes,
                "loja_id": loja["loja_id"],
                "produto_id": prod["produto_id"],
                "estoque_inicial": estoque_inicial,
                "estoque_alvo": estoque_alvo,
                "dias_ruptura_mes": dias_ruptura
            })

fato_estoque = pd.DataFrame(estoque_rows)
fato_estoque.to_parquet(f"{RAW_DIR}/fato_estoque.parquet", index=False)

print(fato_estoque.shape)
fato_estoque.head(10)

(3840, 6)


,mes_referencia,loja_id,produto_id,estoque_inicial,estoque_alvo,dias_ruptura_mes
0,2024-01-01,L01,P001,95,70,1
1,2024-02-01,L01,P001,50,70,0
2,2024-03-01,L01,P001,56,70,1
3,2024-04-01,L01,P001,94,70,0
4,2024-05-01,L01,P001,49,70,0
5,2024-06-01,L01,P001,69,70,0
6,2024-07-01,L01,P001,48,70,0
7,2024-08-01,L01,P001,77,70,3
8,2024-09-01,L01,P001,63,70,5
9,2024-10-01,L01,P001,39,70,0


In [ ]:
# ==== FATO: INVESTIMENTO EM MARKETING ====
canais_marketing = ["Redes Sociais", "Panfletagem", "Rádio Local", "Encarte Promocional"]

marketing_rows = []

for _, loja in lojas.iterrows():
    # nível de investimento "padrão" da loja (algumas investem mais que outras)
    fator_investimento_loja = np.random.uniform(0.6, 1.6)

    for mes in meses:
        for canal_mkt in canais_marketing:
            # nem todo canal é usado em todo mês/loja
            if np.random.rand() > 0.65:
                continue

            valor_base = np.random.uniform(500, 4000)
            valor_investido = round(valor_base * fator_investimento_loja, 2)

            marketing_rows.append({
                "mes_referencia": mes,
                "loja_id": loja["loja_id"],
                "canal_marketing": canal_mkt,
                "valor_investido": valor_investido
            })

fato_marketing = pd.DataFrame(marketing_rows)
fato_marketing.to_parquet(f"{RAW_DIR}/fato_marketing.parquet", index=False)

print(fato_marketing.shape)
fato_marketing.head(10)

(316, 4)


,mes_referencia,loja_id,canal_marketing,valor_investido
0,2024-01-01,L01,Panfletagem,1913.25
1,2024-01-01,L01,Rádio Local,3495.78
2,2024-01-01,L01,Encarte Promocional,4436.38
3,2024-02-01,L01,Panfletagem,2586.53
4,2024-02-01,L01,Rádio Local,4710.94
5,2024-02-01,L01,Encarte Promocional,1748.35
6,2024-03-01,L01,Redes Sociais,1699.69
7,2024-03-01,L01,Encarte Promocional,1574.06
8,2024-04-01,L01,Redes Sociais,1282.98
9,2024-04-01,L01,Rádio Local,3905.24


In [ ]:
# ==== VALIDAÇÃO DE QUALIDADE — CAMADA RAW ====

tabelas = {
    "dim_lojas": lojas,
    "dim_categorias": categorias,
    "dim_produtos": produtos,
    "fato_vendas": fato_vendas,
    "fato_estoque": fato_estoque,
    "fato_marketing": fato_marketing,
}

print("=" * 60)
for nome, df in tabelas.items():
    print(f"\n📋 Tabela: {nome}")
    print(f"Shape: {df.shape}")
    print(f"Nulos por coluna:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    duplicadas = df.duplicated().sum()
    print(f"Linhas duplicadas: {duplicadas}")
    print(f"Tipos de dados:\n{df.dtypes}")
    print("-" * 60)


📋 Tabela: dim_lojas
Shape: (10, 4)
Nulos por coluna:
Series([], dtype: int64)
Linhas duplicadas: 0
Tipos de dados:
loja_id      object
nome_loja    object
regiao       object
cidade       object
dtype: object
------------------------------------------------------------

📋 Tabela: dim_categorias
Shape: (8, 2)
Nulos por coluna:
Series([], dtype: int64)
Linhas duplicadas: 0
Tipos de dados:
categoria_id    object
categoria       object
dtype: object
------------------------------------------------------------

📋 Tabela: dim_produtos
Shape: (32, 5)
Nulos por coluna:
Series([], dtype: int64)
Linhas duplicadas: 0
Tipos de dados:
produto_id       object
categoria_id     object
nome_produto     object
preco_base      float64
custo_base      float64
dtype: object
------------------------------------------------------------

📋 Tabela: fato_vendas
Shape: (76688, 8)
Nulos por coluna:
Series([], dtype: int64)
Linhas duplicadas: 0
Tipos de dados:
data               datetime64[ns]
loja_id            

In [ ]:
# ==== PROCESSED: ENRIQUECER FATO_VENDAS ====

fato_vendas_proc = fato_vendas.copy()
fato_vendas_proc["data"] = pd.to_datetime(fato_vendas_proc["data"])
fato_vendas_proc["mes_referencia"] = fato_vendas_proc["data"].values.astype("datetime64[M]")

# joins com dimensões
fato_vendas_proc = fato_vendas_proc.merge(produtos[["produto_id", "categoria_id", "nome_produto", "custo_base"]], on="produto_id", how="left")
fato_vendas_proc = fato_vendas_proc.merge(categorias, on="categoria_id", how="left")
fato_vendas_proc = fato_vendas_proc.merge(lojas[["loja_id", "nome_loja", "regiao", "cidade"]], on="loja_id", how="left")

# campos derivados estruturais
fato_vendas_proc["valor_total_venda"] = (fato_vendas_proc["quantidade"] * fato_vendas_proc["preco_final"]).round(2)
fato_vendas_proc["custo_total"] = (fato_vendas_proc["quantidade"] * fato_vendas_proc["custo_base"]).round(2)

# tipos padronizados
for col in ["loja_id", "produto_id", "categoria_id", "canal"]:
    fato_vendas_proc[col] = fato_vendas_proc[col].astype("string")

fato_vendas_proc.to_parquet(f"{PROCESSED_DIR}/fato_vendas_processed.parquet", index=False)

print(fato_vendas_proc.shape)
fato_vendas_proc.head(5)

(76688, 18)


,data,loja_id,produto_id,quantidade,preco_praticado,desconto_pct,preco_final,canal,mes_referencia,categoria_id,nome_produto,custo_base,categoria,nome_loja,regiao,cidade,valor_total_venda,custo_total
0,2024-01-01,L01,P001,2,30.13,0.0,30.13,Loja Física,2024-01-01,C01,Arroz 5kg,16.5,Mercearia,Sabor & Cia - Unidade 1,Capital,São Paulo,60.26,33.0
1,2024-01-02,L01,P001,4,27.57,0.0,27.57,Loja Física,2024-01-01,C01,Arroz 5kg,16.5,Mercearia,Sabor & Cia - Unidade 1,Capital,São Paulo,110.28,66.0
2,2024-01-03,L01,P001,4,27.50,0.0,27.50,Loja Física,2024-01-01,C01,Arroz 5kg,16.5,Mercearia,Sabor & Cia - Unidade 1,Capital,São Paulo,110.00,66.0
3,2024-01-04,L01,P001,5,28.77,0.0,28.77,Loja Física,2024-01-01,C01,Arroz 5kg,16.5,Mercearia,Sabor & Cia - Unidade 1,Capital,São Paulo,143.85,82.5
4,2024-01-05,L01,P001,3,27.54,0.0,27.54,Loja Física,2024-01-01,C01,Arroz 5kg,16.5,Mercearia,Sabor & Cia - Unidade 1,Capital,São Paulo,82.62,49.5


In [ ]:
# Validação: nulos nas colunas trazidas pelos merges
colunas_dimensao = [c for c in fato_vendas_proc.columns if c not in fato_vendas.columns]
print(fato_vendas_proc[colunas_dimensao].isnull().sum())

mes_referencia       0
categoria_id         0
nome_produto         0
custo_base           0
categoria            0
nome_loja            0
regiao               0
cidade               0
valor_total_venda    0
custo_total          0
dtype: int64


In [ ]:
print(fato_estoque.dtypes)
print(fato_estoque.head(3))

mes_referencia      datetime64[ns]
loja_id                     object
produto_id                  object
estoque_inicial              int64
estoque_alvo                 int64
dias_ruptura_mes             int64
dtype: object
  mes_referencia loja_id produto_id  estoque_inicial  estoque_alvo  \
0     2024-01-01     L01       P001               95            70   
1     2024-02-01     L01       P001               50            70   
2     2024-03-01     L01       P001               56            70   

   dias_ruptura_mes  
0                 1  
1                 0  
2                 1  


In [ ]:
import pandas as pd

RAW_DIR = "/content/custo-vida-brasil/data/raw"
PROCESSED_DIR = "/content/custo-vida-brasil/data/processed"

dim_lojas = pd.read_parquet(f"{RAW_DIR}/dim_lojas.parquet")
dim_categorias = pd.read_parquet(f"{RAW_DIR}/dim_categorias.parquet")
dim_produtos = pd.read_parquet(f"{RAW_DIR}/dim_produtos.parquet")
fato_vendas = pd.read_parquet(f"{RAW_DIR}/fato_vendas.parquet")
fato_estoque = pd.read_parquet(f"{RAW_DIR}/fato_estoque.parquet")
fato_marketing = pd.read_parquet(f"{RAW_DIR}/fato_marketing.parquet")

print("Raw recarregado:", dim_lojas.shape, dim_categorias.shape, dim_produtos.shape,
      fato_vendas.shape, fato_estoque.shape, fato_marketing.shape)

Raw recarregado: (10, 4) (8, 2) (32, 5) (76688, 8) (3840, 6) (316, 4)


In [ ]:
%who_ls DataFrame

['categorias',
 'comparacao_ruptura_completa',
 'comparativo_margem',
 'comparativo_produtos',
 'custo_produto_mes_tabela',
 'custo_produto_regiao_tabela',
 'desconto_mes',
 'df',
 'df_hip',
 'df_nulos',
 'df_plot',
 'dim_categorias',
 'dim_categorias_proc',
 'dim_lojas',
 'dim_lojas_proc',
 'dim_produtos',
 'dim_produtos_proc',
 'fato_estoque',
 'fato_estoque_proc',
 'fato_marketing',
 'fato_marketing_proc',
 'fato_vendas',
 'fato_vendas_proc',
 'faturamento_mes_faixa',
 'ipca',
 'ipca_desconto',
 'ipca_raw',
 'lojas',
 'margem_canal_mes',
 'margem_com_faixa',
 'margem_com_ruptura',
 'margem_loja_produto_mes',
 'margem_media_produto',
 'margem_mensal_com_ruido',
 'margem_produto_mes',
 'margem_q1',
 'margem_q4',
 'produtos',
 'resumo_hipoteses',
 'variacao_maxima_pp']

In [ ]:
# PROCESSED: PADRONIZAR FATO_ESTOQUE

fato_estoque_proc = fato_estoque.copy()

# Enriquecer com dimensões
fato_estoque_proc = fato_estoque_proc.merge(dim_lojas, on="loja_id", how="left")
fato_estoque_proc = fato_estoque_proc.merge(dim_produtos, on="produto_id", how="left")

# Tipos padronizados
for col in ["loja_id", "produto_id", "categoria_id"]:
    if col in fato_estoque_proc.columns:
        fato_estoque_proc[col] = fato_estoque_proc[col].astype("category")

# Salvar
fato_estoque_proc.to_parquet(f"{PROCESSED_DIR}/fato_estoque_proc.parquet", index=False)

print(fato_estoque_proc.shape)
print(fato_estoque_proc.dtypes)

(3840, 13)
mes_referencia      datetime64[ns]
loja_id                   category
produto_id                category
estoque_inicial              int64
estoque_alvo                 int64
dias_ruptura_mes             int64
nome_loja                   object
regiao                      object
cidade                      object
categoria_id              category
nome_produto                object
preco_base                 float64
custo_base                 float64
dtype: object


In [ ]:
colunas_dimensao = [c for c in fato_estoque_proc.columns if c not in fato_estoque.columns]
print(fato_estoque_proc[colunas_dimensao].isnull().sum())
print(fato_estoque_proc.shape[0] == fato_estoque.shape[0])
print(fato_estoque_proc.shape)

nome_loja       0
regiao          0
cidade          0
categoria_id    0
nome_produto    0
preco_base      0
custo_base      0
dtype: int64
True
(3840, 13)


In [ ]:
print(fato_marketing.dtypes)
print(fato_marketing.head(3))

mes_referencia     datetime64[ns]
loja_id                    object
canal_marketing            object
valor_investido           float64
dtype: object
  mes_referencia loja_id      canal_marketing  valor_investido
0     2024-01-01     L01          Panfletagem          1913.25
1     2024-01-01     L01          Rádio Local          3495.78
2     2024-01-01     L01  Encarte Promocional          4436.38


In [ ]:
# PROCESSED: PADRONIZAR FATO_MARKETING

fato_marketing_proc = fato_marketing.copy()

# Enriquecer com dimensão de lojas
fato_marketing_proc = fato_marketing_proc.merge(dim_lojas, on="loja_id", how="left")

# Tipos padronizados
fato_marketing_proc["loja_id"] = fato_marketing_proc["loja_id"].astype("category")
fato_marketing_proc["canal_marketing"] = fato_marketing_proc["canal_marketing"].astype("category")

# Salvar
fato_marketing_proc.to_parquet(f"{PROCESSED_DIR}/fato_marketing_proc.parquet", index=False)

print(fato_marketing_proc.shape)
print(fato_marketing_proc.dtypes)

(316, 7)
mes_referencia     datetime64[ns]
loja_id                  category
canal_marketing          category
valor_investido           float64
nome_loja                  object
regiao                     object
cidade                     object
dtype: object


In [ ]:
colunas_dimensao = [c for c in fato_marketing_proc.columns if c not in fato_marketing.columns]
print(fato_marketing_proc[colunas_dimensao].isnull().sum())
print(fato_marketing_proc.shape[0] == fato_marketing.shape[0])
print(fato_marketing_proc.shape)

nome_loja    0
regiao       0
cidade       0
dtype: int64
True
(316, 7)


## 2. Construção dos indicadores e teste das hipóteses

### Hipótese 4 — Mudança de mix de produtos (alta → baixa margem)

Classificamos os produtos por faixa de margem e cruzamos com o faturamento mensal, pra checar se uma mudança na composição de vendas (mais produtos de baixa margem sendo vendidos) explicaria a queda percebida.

In [ ]:
# Agrupa por produto e mês, somando faturamento e custo
margem_produto_mes = (
    fato_vendas_proc
    .groupby(['produto_id', 'nome_produto', 'mes_referencia'])
    .agg(
        faturamento=('valor_total_venda', 'sum'),
        custo=('custo_total', 'sum')
    )
    .reset_index()
)

# Calcula margem % = (faturamento - custo) / faturamento
margem_produto_mes['margem_pct'] = (
    (margem_produto_mes['faturamento'] - margem_produto_mes['custo'])
    / margem_produto_mes['faturamento']
) * 100

margem_produto_mes.head(10)

,produto_id,nome_produto,mes_referencia,faturamento,custo,margem_pct
0,P001,Arroz 5kg,2024-01-01,28735.00,18166.5,36.779189
1,P001,Arroz 5kg,2024-02-01,31428.27,19932.0,36.579392
2,P001,Arroz 5kg,2024-03-01,32261.36,20212.5,37.347651
3,P001,Arroz 5kg,2024-04-01,31697.31,19668.0,37.950571
4,P001,Arroz 5kg,2024-05-01,30069.96,18826.5,37.391004
5,P001,Arroz 5kg,2024-06-01,29918.66,18711.0,37.460434
6,P001,Arroz 5kg,2024-07-01,24161.83,15180.0,37.173633
7,P001,Arroz 5kg,2024-08-01,24235.64,15081.0,37.773461
8,P001,Arroz 5kg,2024-09-01,22039.88,14091.0,36.065895
9,P001,Arroz 5kg,2024-10-01,22761.86,14388.0,36.788997


In [ ]:
import plotly.express as px

# Seleciona alguns produtos de categorias diferentes pra comparar
produtos_amostra = margem_produto_mes['produto_id'].unique()[:6]  # ajuste os IDs se quiser produtos específicos

df_plot = margem_produto_mes[margem_produto_mes['produto_id'].isin(produtos_amostra)]

fig = px.line(
    df_plot,
    x='mes_referencia',
    y='margem_pct',
    color='nome_produto',
    title='Margem % por produto ao longo de 2024',
    markers=True
)
fig.update_layout(yaxis_title='Margem (%)', xaxis_title='Mês')
fig.show()

In [ ]:
# Cria coluna de trimestre a partir do mês
margem_produto_mes['trimestre'] = pd.PeriodIndex(margem_produto_mes['mes_referencia'], freq='M').quarter

# Margem média por produto no 1º trimestre (Jan-Mar) e no 4º trimestre (Out-Dez)
margem_q1 = (
    margem_produto_mes[margem_produto_mes['trimestre'] == 1]
    .groupby(['produto_id', 'nome_produto'])['margem_pct']
    .mean()
    .reset_index(name='margem_pct_q1')
)

margem_q4 = (
    margem_produto_mes[margem_produto_mes['trimestre'] == 4]
    .groupby(['produto_id', 'nome_produto'])['margem_pct']
    .mean()
    .reset_index(name='margem_pct_q4')
)

# Junta as duas tabelas e calcula a variação
comparativo_margem = margem_q1.merge(margem_q4, on=['produto_id', 'nome_produto'])
comparativo_margem['variacao_pp'] = comparativo_margem['margem_pct_q4'] - comparativo_margem['margem_pct_q1']

# Ordena do que mais caiu pro que mais subiu
comparativo_margem = comparativo_margem.sort_values('variacao_pp')

comparativo_margem

,produto_id,nome_produto,margem_pct_q1,margem_pct_q4,variacao_pp
31,P032,Torta Salgada,48.007925,47.348521,-0.659404
19,P020,Água Sanitária 1L,46.096310,45.521871,-0.574439
2,P003,Açúcar 1kg,39.198407,38.625272,-0.573135
13,P014,Queijo Mussarela kg,37.334206,36.796859,-0.537347
20,P021,Sabonete unid,51.186733,50.684802,-0.501931
23,P024,Creme Dental 90g,44.491192,44.029126,-0.462065
3,P004,Óleo de Soja 900ml,38.453497,38.044737,-0.408760
28,P029,Pão Francês kg,52.886325,52.492109,-0.394216
29,P030,Bolo Fatia,55.311986,54.967089,-0.344897
25,P026,Cerveja Lata,42.033751,41.692361,-0.341390


In [ ]:
# Margem média anual de cada produto (base pra classificação)
margem_media_produto = (
    margem_produto_mes
    .groupby(['produto_id', 'nome_produto'])['margem_pct']
    .mean()
    .reset_index(name='margem_pct_media')
)

# Classifica em Alta/Baixa pela mediana da margem média dos produtos
mediana_margem = margem_media_produto['margem_pct_media'].median()

margem_media_produto['faixa_margem'] = margem_media_produto['margem_pct_media'].apply(
    lambda x: 'Alta' if x >= mediana_margem else 'Baixa'
)

print(f"Mediana usada como corte: {mediana_margem:.2f}%")
margem_media_produto.sort_values('margem_pct_media', ascending=False)

Mediana usada como corte: 41.78%


,produto_id,nome_produto,margem_pct_media,faixa_margem
29,P030,Bolo Fatia,55.246325,Alta
6,P007,Alface unid,54.914616,Alta
28,P029,Pão Francês kg,52.684880,Alta
20,P021,Sabonete unid,50.814112,Alta
30,P031,Rosquinha 300g,49.995004,Alta
4,P005,Banana Prata kg,48.958862,Alta
26,P027,"Água Mineral 1,5L",48.749682,Alta
5,P006,Tomate kg,48.450197,Alta
16,P017,Detergente 500ml,48.224042,Alta
31,P032,Torta Salgada,47.645865,Alta


In [ ]:
# Junta a classificação de faixa de margem com o faturamento mensal por produto
margem_com_faixa = margem_produto_mes.merge(
    margem_media_produto[['produto_id', 'faixa_margem']],
    on='produto_id'
)

# Soma o faturamento por mês e faixa de margem
faturamento_mes_faixa = (
    margem_com_faixa
    .groupby(['mes_referencia', 'faixa_margem'])['faturamento']
    .sum()
    .reset_index()
)

# Calcula o % que cada faixa representa do faturamento total do mês
faturamento_mes_faixa['pct_do_mes'] = (
    faturamento_mes_faixa
    .groupby('mes_referencia')['faturamento']
    .transform(lambda x: x / x.sum() * 100)
)

faturamento_mes_faixa

,mes_referencia,faixa_margem,faturamento,pct_do_mes
0,2024-01-01,Alta,119279.75,25.666726
1,2024-01-01,Baixa,345445.48,74.333274
2,2024-02-01,Alta,115962.45,25.223974
3,2024-02-01,Baixa,343768.64,74.776026
4,2024-03-01,Alta,129641.51,26.187681
5,2024-03-01,Baixa,365406.18,73.812319
6,2024-04-01,Alta,119710.39,25.512781
7,2024-04-01,Baixa,349506.94,74.487219
8,2024-05-01,Alta,117824.06,25.532590
9,2024-05-01,Baixa,343641.31,74.467410


In [ ]:
# Margem por loja + produto + mês (mais granular que antes)
margem_loja_produto_mes = (
    fato_vendas_proc
    .groupby(['loja_id', 'produto_id', 'nome_produto', 'mes_referencia'])
    .agg(
        faturamento=('valor_total_venda', 'sum'),
        custo=('custo_total', 'sum')
    )
    .reset_index()
)

margem_loja_produto_mes['margem_pct'] = (
    (margem_loja_produto_mes['faturamento'] - margem_loja_produto_mes['custo'])
    / margem_loja_produto_mes['faturamento']
) * 100

# Junta com os dias de ruptura da fato_estoque_proc
margem_com_ruptura = margem_loja_produto_mes.merge(
    fato_estoque_proc[['loja_id', 'produto_id', 'mes_referencia', 'dias_ruptura_mes']],
    on=['loja_id', 'produto_id', 'mes_referencia'],
    how='left'
)

margem_com_ruptura.head(10)

,loja_id,produto_id,nome_produto,mes_referencia,faturamento,custo,margem_pct,dias_ruptura_mes
0,L01,P001,Arroz 5kg,2024-01-01,3317.40,2079.0,37.330440,1
1,L01,P001,Arroz 5kg,2024-02-01,4098.06,2689.5,34.371385,0
2,L01,P001,Arroz 5kg,2024-03-01,3932.06,2491.5,36.636267,1
3,L01,P001,Arroz 5kg,2024-04-01,3351.29,2145.0,35.994796,0
4,L01,P001,Arroz 5kg,2024-05-01,3534.66,2145.0,39.315238,0
5,L01,P001,Arroz 5kg,2024-06-01,3704.73,2293.5,38.092655,0
6,L01,P001,Arroz 5kg,2024-07-01,2446.04,1567.5,35.916829,0
7,L01,P001,Arroz 5kg,2024-08-01,2874.21,1848.0,35.704072,3
8,L01,P001,Arroz 5kg,2024-09-01,2653.75,1683.0,36.580311,5
9,L01,P001,Arroz 5kg,2024-10-01,2933.08,1848.0,36.994559,0


In [ ]:
# Quantos NaN existem e qual % da tabela isso representa
total_linhas = len(margem_com_ruptura)
nulos = margem_com_ruptura['dias_ruptura_mes'].isnull().sum()

print(f"Total de linhas: {total_linhas}")
print(f"Linhas com NaN em dias_ruptura_mes: {nulos}")
print(f"Percentual: {nulos / total_linhas * 100:.2f}%")

Total de linhas: 3840
Linhas com NaN em dias_ruptura_mes: 0
Percentual: 0.00%


In [ ]:
# Filtra só as linhas com NaN
df_nulos = margem_com_ruptura[margem_com_ruptura['dias_ruptura_mes'].isnull()]

# Ruptura de NaN por loja
print("NaN por loja:")
print(df_nulos['loja_id'].value_counts())

# NaN por mês
print("\nNaN por mês:")
print(df_nulos['mes_referencia'].value_counts())

# NaN por produto
print("\nNaN por produto:")
print(df_nulos['produto_id'].value_counts())

NaN por loja:
Series([], Name: count, dtype: int64)

NaN por mês:
Series([], Name: count, dtype: int64)

NaN por produto:
Series([], Name: count, dtype: int64)


In [ ]:
print(f"Linhas em margem_loja_produto_mes (antes do merge): {len(margem_loja_produto_mes)}")
print(f"Linhas em margem_com_ruptura (depois do merge): {len(margem_com_ruptura)}")

Linhas em margem_loja_produto_mes (antes do merge): 3840
Linhas em margem_com_ruptura (depois do merge): 3840


### Hipótese 5 — Ruptura de estoque causando reposição emergencial mais cara

Mecanismo testado: ruptura de estoque → reposição emergencial fora do desconto por volume → custo de aquisição mais alto naquele mês → margem menor. Comparamos a margem média em meses com e sem ruptura registrada.

In [ ]:
import numpy as np

margem_com_ruptura['teve_ruptura'] = np.where(
    margem_com_ruptura['dias_ruptura_mes'] > 0, 'Sim', 'Não'
)

In [ ]:
print(margem_com_ruptura['teve_ruptura'].value_counts())

teve_ruptura
Sim    2111
Não    1729
Name: count, dtype: int64


In [ ]:
comparacao_ruptura_completa = margem_com_ruptura.groupby('teve_ruptura')['margem_pct'].agg(['mean', 'median', 'std', 'count'])
print(comparacao_ruptura_completa)

                   mean     median       std  count
teve_ruptura                                       
Não           43.022373  41.975012  6.171243   1729
Sim           43.117372  42.134407  6.124038   2111


### Hipóteses 3 e 6 — Custo do fornecedor e logística regional

Analisamos o custo médio por produto, segmentado por região e por mês, pra identificar se houve aumento de custo do fornecedor sem repasse ao preço final, ou se lojas do interior sofrem com frete/logística mais cara.

In [ ]:
print(fato_vendas_proc.columns.tolist())

['data', 'loja_id', 'produto_id', 'quantidade', 'preco_praticado', 'desconto_pct', 'preco_final', 'canal', 'mes_referencia', 'categoria_id', 'nome_produto', 'custo_base', 'categoria', 'nome_loja', 'regiao', 'cidade', 'valor_total_venda', 'custo_total']


In [ ]:
fato_vendas_proc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76688 entries, 0 to 76687
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   data               76688 non-null  datetime64[ns]
 1   loja_id            76688 non-null  string        
 2   produto_id         76688 non-null  string        
 3   quantidade         76688 non-null  int64         
 4   preco_praticado    76688 non-null  float64       
 5   desconto_pct       76688 non-null  float64       
 6   preco_final        76688 non-null  float64       
 7   canal              76688 non-null  string        
 8   mes_referencia     76688 non-null  datetime64[s] 
 9   categoria_id       76688 non-null  string        
 10  nome_produto       76688 non-null  object        
 11  custo_base         76688 non-null  float64       
 12  categoria          76688 non-null  object        
 13  nome_loja          76688 non-null  object        
 14  regiao

In [ ]:
custo_produto_regiao = fato_vendas_proc.groupby(['produto_id', 'regiao'])['custo_base'].mean()
print(custo_produto_regiao)

produto_id  regiao   
P001        Capital      16.5
            Grande SP    16.5
            Interior     16.5
            Litoral      16.5
P002        Capital       5.8
                         ... 
P031        Litoral       4.2
P032        Capital       5.5
            Grande SP     5.5
            Interior      5.5
            Litoral       5.5
Name: custo_base, Length: 128, dtype: float64


In [ ]:
custo_produto_regiao_tabela = custo_produto_regiao.unstack()
print(custo_produto_regiao_tabela)

regiao      Capital  Grande SP  Interior  Litoral
produto_id                                       
P001           16.5       16.5      16.5     16.5
P002            5.8        5.8       5.8      5.8
P003            2.9        2.9       2.9      2.9
P004            5.2        5.2       5.2      5.2
P005            3.2        3.2       3.2      3.2
P006            4.1        4.1       4.1      4.1
P007            1.2        1.2       1.2      1.2
P008            2.8        2.8       2.8      2.8
P009           42.0       42.0      42.0     42.0
P010            8.5        8.5       8.5      8.5
P011           12.3       12.3      12.3     12.3
P012           19.5       19.5      19.5     19.5
P013            3.6        3.6       3.6      3.6
P014           22.0       22.0      22.0     22.0
P015            1.7        1.7       1.7      1.7
P016            6.5        6.5       6.5      6.5
P017            1.6        1.6       1.6      1.6
P018            8.2        8.2       8.2      8.2


In [ ]:
custo_produto_mes = fato_vendas_proc.groupby(['produto_id', 'mes_referencia'])['custo_base'].mean()
print(custo_produto_mes)

custo_produto_mes_tabela = custo_produto_mes.unstack()
print(custo_produto_mes_tabela)

produto_id  mes_referencia
P001        2024-01-01        16.5
            2024-02-01        16.5
            2024-03-01        16.5
            2024-04-01        16.5
            2024-05-01        16.5
                              ... 
P032        2024-08-01         5.5
            2024-09-01         5.5
            2024-10-01         5.5
            2024-11-01         5.5
            2024-12-01         5.5
Name: custo_base, Length: 384, dtype: float64
mes_referencia  2024-01-01  2024-02-01  2024-03-01  2024-04-01  2024-05-01  \
produto_id                                                                   
P001                  16.5        16.5        16.5        16.5        16.5   
P002                   5.8         5.8         5.8         5.8         5.8   
P003                   2.9         2.9         2.9         2.9         2.9   
P004                   5.2         5.2         5.2         5.2         5.2   
P005                   3.2         3.2         3.2         3.2         3.2

In [ ]:
print(custo_produto_mes_tabela.head())

mes_referencia  2024-01-01  2024-02-01  2024-03-01  2024-04-01  2024-05-01  \
produto_id                                                                   
P001                  16.5        16.5        16.5        16.5        16.5   
P002                   5.8         5.8         5.8         5.8         5.8   
P003                   2.9         2.9         2.9         2.9         2.9   
P004                   5.2         5.2         5.2         5.2         5.2   
P005                   3.2         3.2         3.2         3.2         3.2   

mes_referencia  2024-06-01  2024-07-01  2024-08-01  2024-09-01  2024-10-01  \
produto_id                                                                   
P001                  16.5        16.5        16.5        16.5        16.5   
P002                   5.8         5.8         5.8         5.8         5.8   
P003                   2.9         2.9         2.9         2.9         2.9   
P004                   5.2         5.2         5.2         5.2 

In [ ]:
desconto_mes = fato_vendas_proc.groupby('mes_referencia')['desconto_pct'].mean()
print(desconto_mes)

mes_referencia
2024-01-01    0.033902
2024-02-01    0.034844
2024-03-01    0.035789
2024-04-01    0.036738
2024-05-01    0.036273
2024-06-01    0.035591
2024-07-01    0.035761
2024-08-01    0.037313
2024-09-01    0.036948
2024-10-01    0.035307
2024-11-01    0.036395
2024-12-01    0.035726
Name: desconto_pct, dtype: float64


### Hipótese 7 — Repasse de inflação (IPCA) via aumento de desconto

Buscamos o IPCA mensal via API pública do SIDRA/IBGE e testamos a correlação entre a variação da inflação e o desconto médio praticado pela rede.

In [ ]:
!pip install sidrapy

In [ ]:
import sidrapy
import pandas as pd

# Etapa 2 + 3: buscar o IPCA de 2024 com tratamento de erro
try:
    ipca_raw = sidrapy.get_table(
        table_code='1737',
        territorial_level='1',
        ibge_territorial_code='all',
        variable='63',  # variação mensal do IPCA
        period='202401-202412'
    )
    print("Requisição concluída com sucesso.")
except Exception as e:
    print(f"Erro ao buscar dados do IPCA na API do IBGE: {e}")
    ipca_raw = None

In [ ]:
print(ipca_raw)

In [ ]:
# Remove a linha de cabeçalho duplicada (índice 0)
ipca = ipca_raw.drop(index=0).reset_index(drop=True)

# Mantém só as colunas que interessam: mês (D2C) e valor (V)
ipca = ipca[['D2C', 'V']].rename(columns={'D2C': 'mes_referencia', 'V': 'ipca_variacao'})

# Corrige os tipos
ipca['ipca_variacao'] = ipca['ipca_variacao'].astype(float)
ipca['mes_referencia'] = ipca['mes_referencia'].astype(str)  # já vem como '202401' etc.

ipca.head()

In [ ]:
ipca['mes_referencia'] = pd.to_datetime(ipca['mes_referencia'], format='%Y%m')

ipca.head()

In [ ]:
desconto_mes = desconto_mes.reset_index()

desconto_mes.head()

In [ ]:
ipca_desconto = pd.merge(ipca, desconto_mes, on='mes_referencia', how='left')

ipca_desconto.head(12)

In [ ]:
ipca_desconto[['ipca_variacao', 'desconto_pct']].corr()

### Hipótese 8 — Canal de venda (Delivery/App vs. Loja Física) impactando margem

Comparamos a margem média mensal entre os canais de venda (loja física vs. delivery/app), pra verificar se a mudança no mix de canais explicaria parte da variação de margem.

In [ ]:
fato_vendas_proc['canal'].unique()

In [ ]:
fato_vendas_proc.columns.tolist()

In [ ]:
margem_loja_produto_mes.columns.tolist()

In [ ]:
fato_vendas_proc['margem_pct'] = (
    (fato_vendas_proc['valor_total_venda'] - fato_vendas_proc['custo_total'])
    / fato_vendas_proc['valor_total_venda']
)

In [ ]:
margem_canal_mes = fato_vendas_proc.groupby(['mes_referencia', 'canal'])['margem_pct'].mean().unstack()

margem_canal_mes

In [ ]:
margem_canal_mes['diferenca'] = margem_canal_mes['Loja Física'] - margem_canal_mes['Delivery/App']

margem_canal_mes

In [ ]:
margem_mes = fato_vendas_proc.groupby('mes_referencia')['margem_pct'].mean()

margem_mes

## 3. Consolidando a margem mensal e o benchmark de ruído

Com todas as hipóteses individuais testadas, consolidamos a margem média mensal da rede inteira e calculamos a variação mês a mês por produto — a base para o benchmark de ruído natural do negócio (validado empiricamente, não assumido).

In [ ]:
import plotly.express as px

fig = px.line(
    margem_mes.reset_index(),
    x='mes_referencia',
    y='margem_pct',
    title='Margem média mensal — 2024',
    markers=True
)
fig.show()

In [ ]:
print(comparativo_margem.describe())

In [ ]:
# variação mês a mês, por produto
variacao_por_produto = margem_produto_mes.groupby('produto_id')['margem_pct'].apply(lambda x: x.diff().abs().max())
print(variacao_por_produto.describe())

## 4. Descoberta central e recomendação

**Achado:** a margem nunca caiu de fato. A oscilação real observada (~0.4pp) está bem dentro do ruído natural do negócio (~1.27pp — a variação média mês a mês por produto, calculada empiricamente e não assumida a priori).

**Causa da percepção:** o alarme surgiu ao analisar um único trimestre isoladamente (Q2, cujo mês mais baixo do ano concentrava a preocupação), sem comparação com uma faixa de variação normal. Um gráfico com eixo Y truncado reforçou essa leitura distorcida.

**Recomendação:** adotar 1.27pp como benchmark oficial de ruído para análises futuras de margem, evitando alarmes falsos e decisões precipitadas baseadas em oscilações normais do negócio.

As tabelas abaixo são exportadas para a camada analytics e consomem o dashboard interativo publicado.

In [ ]:
import os

# Média geral da margem no ano (nível macro, pra centralizar a faixa)
media_margem_ano = margem_mes.mean()
ruido_natural = 1.27  # pp, calculado como variação média mês a mês por produto

margem_mensal_com_ruido = pd.DataFrame({
    'mes_referencia': margem_mes.index,
    'margem_pct': margem_mes.values * 100,           # convertendo pra pp, mais legível no dashboard
    'media_ano_pct': media_margem_ano * 100,
    'limite_superior_pct': (media_margem_ano * 100) + ruido_natural,
    'limite_inferior_pct': (media_margem_ano * 100) - ruido_natural,
})

margem_mensal_com_ruido = margem_mensal_com_ruido.reset_index(drop=True)

print(margem_mensal_com_ruido)

In [ ]:
caminho = os.path.join(ANALYTICS_DIR, 'margem_mensal_com_ruido.parquet')
margem_mensal_com_ruido.to_parquet(caminho, index=False)

print(f"Salvo em: {caminho}")
print(f"Linhas: {len(margem_mensal_com_ruido)}")

In [ ]:
resumo_hipoteses = pd.DataFrame({
    'numero': [1, 2, 3, 4, 5, 6, 7, 8, 9],
    'hipotese': [
        'Desconto concentrado em produtos/períodos específicos',
        'Marketing ineficiente (baixo investimento gerando queda de margem)',
        'Custo do fornecedor subindo sem repasse ao preço',
        'Mudança de mix de produtos (Alta → Baixa margem)',
        'Ruptura de estoque causando reposição emergencial mais cara',
        'Logística/frete mais caro em lojas do interior',
        'Repasse de inflação (IPCA) via aumento de desconto',
        'Canal de venda (Delivery/App vs. Loja Física) impactando margem',
        'Sazonalidade (datas comemorativas afetando margem)',
    ],
    'status': [
        'Refutada', 'Refutada', 'Refutada', 'Refutada', 'Refutada',
        'Não-testável', 'Refutada', 'Refutada', 'Refutada/Não-testável',
    ],
    'detalhe': [
        None,
        'Correlação espúria (cidades maiores geram mais receita, independente do investimento em marketing)',
        'Variação Q1 vs Q4 dentro do ruído natural por produto (~1.27pp)',
        'Share de faturamento estável 25-26% no ano',
        'Diferença de margem 0.10pp, direção oposta à prevista',
        'custo_base idêntico em todas as regiões',
        'Correlação negativa -0.48, direção oposta ao previsto; amostra pequena (n=12)',
        'Diferença máxima < 1pp, dentro do ruído natural esperado',
        'Amplitude anual de 0.4pp; apenas 1 ano de dados disponível',
    ],
})

print(resumo_hipoteses)

In [ ]:
caminho = os.path.join(ANALYTICS_DIR, 'resumo_hipoteses.parquet')
resumo_hipoteses.to_parquet(caminho, index=False)

print(f"Salvo em: {caminho}")
print(f"Linhas: {len(resumo_hipoteses)}")

In [ ]:
variacao_maxima_pp = variacao_por_produto.reset_index()
variacao_maxima_pp.merge(dim_produtos,on='produto_id')
comparativo_produtos = variacao_maxima_pp.merge(dim_produtos, on='produto_id')
print(comparativo_produtos)

In [ ]:
comparativo_produtos = comparativo_produtos.rename(columns={'margem_pct': 'variacao_maxima_pp'})
print(comparativo_produtos.columns.tolist())

In [ ]:
caminho = os.path.join(ANALYTICS_DIR, 'comparativo_produtos.parquet')
comparativo_produtos.to_parquet(caminho, index=False)

print(f"Salvo em: {caminho}")
print(f"Linhas: {len(comparativo_produtos)}")

---

## Impacto esperado e próximos passos

**Impacto esperado:** ao adotar o benchmark de ruído validado, a gestão evita investir tempo e recursos corrigindo um "problema" que não existe, redirecionando o foco para oportunidades reais identificadas durante a investigação (ex.: produtos com variação genuinamente acima do ruído, como Carne Moída, que merecem atenção pontual).

**Próximos passos:**
- Expandir a série histórica (múltiplos anos) para testar sazonalidade de forma robusta
- Adicionar testes automatizados sobre as etapas de validação do pipeline
- Monitorar continuamente a margem contra o benchmark de 1.27pp via dashboard publicado

🔗 Dashboard interativo: [margem.streamlit.app](https://margem.streamlit.app)
📂 Repositório completo: [github.com/klaytonsi/custo-vida-brasil-margem-varejo-](https://github.com/klaytonsi/custo-vida-brasil-margem-varejo-)